# 01 - Simple Linear Regression End-to-End

This notebook teaches simple linear regression using one input feature: `study_hours`. It moves from intuition to formula, manual OLS, scikit-learn implementation, statsmodels inference, visualization, diagnostics, and student exercises.

## Learning objectives

Students will learn how to:

1. Define a simple regression problem.
2. Visualize the relationship between one feature and one target.
3. Fit a regression model using scikit-learn.
4. Derive slope and intercept manually using OLS formulas.
5. Evaluate the model using MAE, RMSE, and R².
6. Interpret coefficients responsibly.
7. Inspect residuals and understand model limitations.

## 1. Problem statement

We want to predict `exam_score` from `study_hours`.

The simple linear regression equation is:

```text
exam_score = beta_0 + beta_1 * study_hours + error
```

Where:

| Term | Meaning |
|---|---|
| `beta_0` | Intercept: expected score when study hours are zero |
| `beta_1` | Slope: expected score change for one additional study hour |
| `error` | Difference between actual and predicted score |

## 2. Import libraries

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import statsmodels.api as sm

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

## 3. Load the dataset

In [ ]:
DATA_DIR = Path.cwd().parent / 'data' if (Path.cwd().parent / 'data').exists() else Path.cwd() / 'data'
df = pd.read_csv(DATA_DIR / 'simple_linear_student_scores.csv')
df.head()

## 4. Data quality check

Regression should not start until we know the row count, missing values, and numeric ranges.

In [ ]:
pd.DataFrame({
    'column': df.columns,
    'dtype': [str(dtype) for dtype in df.dtypes],
    'missing_count': df.isna().sum().values,
    'n_unique': df.nunique().values
})

In [ ]:
df[['study_hours', 'attendance_pct', 'exam_score']].describe().round(2)

## 5. Visualize the raw relationship

A linear model is sensible only if the relationship is approximately linear. The scatter plot is the first sanity check.

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df['study_hours'], df['exam_score'], alpha=0.8)
plt.title('Study hours vs exam score')
plt.xlabel('Study hours')
plt.ylabel('Exam score')
plt.grid(True, alpha=0.3)
plt.show()

## 6. Train-test split

We fit the model on training data and evaluate it on unseen test data. This avoids judging the model only on examples it already saw.

In [ ]:
X = df[['study_hours']]
y = df['exam_score']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=43
)

print('Training rows:', len(X_train))
print('Testing rows :', len(X_test))

## 7. Fit scikit-learn LinearRegression

`scikit-learn` is preferred for ML workflow: train-test split, pipelines, cross-validation, and production-style prediction.

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

pred = model.predict(X_test)

print(f'Intercept: {model.intercept_:.4f}')
print(f'Slope    : {model.coef_[0]:.4f}')

## 8. Visualize fitted regression line

This plot overlays the fitted model on top of the original data. The line shows the model's average expected prediction.

In [ ]:
x_line = np.linspace(df['study_hours'].min(), df['study_hours'].max(), 100).reshape(-1, 1)
y_line = model.predict(pd.DataFrame(x_line, columns=['study_hours']))

plt.figure(figsize=(7, 5))
plt.scatter(df['study_hours'], df['exam_score'], alpha=0.7, label='Actual data')
plt.plot(x_line.ravel(), y_line, linewidth=2, label='Fitted regression line')
plt.title('Simple linear regression fitted line')
plt.xlabel('Study hours')
plt.ylabel('Exam score')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 9. Evaluate the model

Use multiple metrics because each metric answers a different question.

| Metric | Meaning |
|---|---|
| MAE | Average absolute error in target units |
| RMSE | Error metric that penalizes large errors more |
| R² | Share of target variation explained by the model |

In [ ]:
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

metrics = pd.DataFrame({
    'metric': ['intercept', 'slope', 'MAE', 'RMSE', 'R2'],
    'value': [model.intercept_, model.coef_[0], mae, rmse, r2]
})
metrics

## 10. Manual OLS implementation

For simple regression, the slope and intercept can be calculated directly.

```text
slope = sum((x_i - mean_x)(y_i - mean_y)) / sum((x_i - mean_x)^2)
intercept = mean_y - slope * mean_x
```

In [ ]:
x = X_train['study_hours'].to_numpy()
yt = y_train.to_numpy()

manual_slope = ((x - x.mean()) * (yt - yt.mean())).sum() / ((x - x.mean()) ** 2).sum()
manual_intercept = yt.mean() - manual_slope * x.mean()

pd.DataFrame({
    'implementation': ['manual_formula', 'sklearn'],
    'intercept': [manual_intercept, model.intercept_],
    'slope': [manual_slope, model.coef_[0]]
})

## 11. Prediction table

This table helps students inspect where the model performs well and where it misses.

In [ ]:
prediction_table = X_test.copy()
prediction_table['actual_exam_score'] = y_test.values
prediction_table['predicted_exam_score'] = pred
prediction_table['residual'] = prediction_table['actual_exam_score'] - prediction_table['predicted_exam_score']
prediction_table.sort_values('study_hours').round(2).head(12)

## 12. Residual diagnostics

Residuals are the model errors. A useful regression model should not show obvious residual patterns.

In [ ]:
residuals = y_test - pred

plt.figure(figsize=(7, 5))
plt.scatter(pred, residuals, alpha=0.8)
plt.axhline(0, linewidth=1)
plt.title('Residuals vs predicted values')
plt.xlabel('Predicted exam score')
plt.ylabel('Residual')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
plt.hist(residuals, bins=10, edgecolor='black')
plt.title('Distribution of residuals')
plt.xlabel('Residual')
plt.ylabel('Count')
plt.grid(True, alpha=0.3)
plt.show()

## 13. statsmodels OLS inference

`statsmodels` is used when we want coefficient tables, p-values, confidence intervals, and statistical summaries.

In [ ]:
ols = sm.OLS(y_train, sm.add_constant(X_train)).fit()

pd.DataFrame({
    'coefficient': ols.params,
    'p_value': ols.pvalues,
    'lower_95': ols.conf_int()[0],
    'upper_95': ols.conf_int()[1]
})

## 14. Cross-validation

Cross-validation estimates model stability across multiple train/test splits.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=43)
cv_r2 = cross_val_score(LinearRegression(), X, y, scoring='r2', cv=cv)
cv_mae = -cross_val_score(LinearRegression(), X, y, scoring='neg_mean_absolute_error', cv=cv)

pd.DataFrame({
    'fold': range(1, 6),
    'r2': cv_r2,
    'mae': cv_mae
}).round(4)

## 15. Final interpretation

The slope tells us the expected change in exam score for one additional study hour. In this synthetic dataset, the relationship is positive. Use this wording:

> One additional study hour is associated with an average increase in expected exam score.

Avoid saying study hours *cause* the score change unless the data was collected through a causal experimental design.

## Student exercises

1. Replace `study_hours` with `attendance_pct` and compare metrics.
2. Fit a two-feature model using `study_hours` and `attendance_pct`.
3. Change the train/test split random seed and observe metric stability.
4. Add a new residual plot using `study_hours` on the x-axis.
5. Write a 5-line conclusion explaining model performance and limitations.